In [3]:
import os, pandas as pd, h5py

h5_file_path = r"E:\whisker_asymmetry\Tetanus_silencing\2026_May_TelC_behavior\WA029_0_MRN_TelC6_M_2_25_26_20260506\WA029_0_MRN_TelC6_pre_injection_2026-05-06_001_0001.h5"
print("exists:", os.path.exists(h5_file_path))
print("size:", os.path.getsize(h5_file_path) if os.path.exists(h5_file_path) else "n/a")

try:
    with pd.HDFStore(h5_file_path, "r") as s:
        print("pandas keys:", s.keys())
except Exception as e:
    print("HDFStore open failed:", type(e).__name__, e)

try:
    with h5py.File(h5_file_path, "r") as f:
        print("h5py top-level keys:", list(f.keys()))
except Exception as e:
    print("h5py open failed:", type(e).__name__, e)

exists: True
size: 129861424
pandas keys: []
h5py top-level keys: ['header', 'sweep_0001']


Convert Pulse into TTLs

In [1]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

#h5_file = Path(r"D:\ear_movement_tail_pinch_20260326\tail_pinch_WA024_L_R_tail_pinch_ear_tracking_2026-03-28_0001.h5")
#h5_file = Path(r"E:\whisker_asymmetry\nob_frame_videos_neuralyzer\Opto_TelC_ttls\WA029_0_MRN_TelC6_02_Day7_2026-05-15_001_0001.h5")

h5_file = Path(r"E:\whisker_asymmetry\Tetanus_silencing\2026_May_TelC_behavior\WA029_0_MRN_TelC6_M_2_25_26_20260506\WA029_0_MRN_TelC6_pre_injection_2026-05-06_001_0001.h5")
out_csv = h5_file.with_name(h5_file.stem + "_pulsepal_ttls.csv")


c:\conda_envs\whisker_tracking\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# PulsePal TTL input line (DI1 -> bit 1 in your setup)
PULSEPAL_BIT = 1

with h5py.File(h5_file, "r") as f:
    fs = float(np.array(f["/header/AcquisitionSampleRate"]).reshape(-1)[0])
    digital = np.array(f["/sweep_0001/digitalScans"])[0].astype(np.uint16)

# Estimate camera FPS from camera TTL on DI0
cam = ((digital & (1 << 0)) != 0)
cam_rise = np.flatnonzero((~cam[:-1]) & cam[1:]) + 1
if cam_rise.size < 2:
    raise RuntimeError("Not enough camera rising edges to estimate FPS.")
fps = 1.0 / np.median(np.diff(cam_rise) / fs)
print("Estimated camera FPS:", fps)

# PulsePal TTL decode
ttl = ((digital & (1 << PULSEPAL_BIT)) != 0)

# Rising/falling edges in DAQ samples
rise_idx = np.flatnonzero((~ttl[:-1]) & ttl[1:]) + 1
fall_idx = np.flatnonzero(ttl[:-1] & (~ttl[1:])) + 1

# Pair each rise with next fall
j = np.searchsorted(fall_idx, rise_idx, side="right")
ok = j < fall_idx.size
rise_idx = rise_idx[ok]
fall_idx = fall_idx[j[ok]]
ok2 = fall_idx > rise_idx
rise_idx = rise_idx[ok2]
fall_idx = fall_idx[ok2]

# Convert sample indices to frame indices using estimated FPS
samples_per_frame = fs / fps
rise_frame = np.rint(rise_idx / samples_per_frame).astype(int)
fall_frame = np.rint(fall_idx / samples_per_frame).astype(int)
duration_frames = (fall_frame - rise_frame).astype(int)

df = pd.DataFrame({
    "pulse_index": np.arange(len(rise_idx), dtype=int),
    "rise_sample": rise_idx.astype(int),
    "fall_sample": fall_idx.astype(int),
    "rise_time_s": rise_idx / fs,
    "fall_time_s": fall_idx / fs,
    "duration_s": (fall_idx - rise_idx) / fs,
    "rise_frame": rise_frame,
    "fall_frame": fall_frame,
    "duration_frames": duration_frames,
    "estimated_fps": fps
})

df.to_csv(out_csv, index=False)
print(f"Saved {len(df)} pulses to: {out_csv}")

Estimated camera FPS: 500.0
Saved 71 pulses to: E:\whisker_asymmetry\Tetanus_silencing\2026_May_TelC_behavior\WA029_0_MRN_TelC6_M_2_25_26_20260506\WA029_0_MRN_TelC6_pre_injection_2026-05-06_001_0001_pulsepal_ttls.csv
